# Method 3 — LayoutLM (text + 2D layout)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chundawat-h/Receipt-Invoice-Information-Extraction/blob/main/notebooks/04_layoutlm.ipynb)

**Issue #6** (Hiranya). Fine-tune `microsoft/layoutlm-base-uncased` on the same task as BERT (#5): label every OCR word as `B-/I-COMPANY`, `DATE`, `ADDRESS`, `TOTAL` or `O`. The only difference is that LayoutLM also gets **each word's box** (x0, y0, x1, y1 on a 0–1000 grid), so it can use *where* a word is on the receipt.

To keep the comparison fair, everything else is the same as BERT:
- **Same words and sub-tokens:** both models use the same uncased WordPiece vocabulary (checked in #4).
- **Same hyperparameters:** they are read from `results/bert_hparams.json`, and the notebook stops if anything differs.
- **Same base size:** 12 layers, hidden size 768.

**Run in Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (training takes a few minutes).

Outputs saved to `results/` (same names as BERT, with `layoutlm_` in front):

| File | Contents |
|---|---|
| `layoutlm_preds.json` | field predictions per receipt key, the input to `evaluate()` and #7 |
| `layoutlm_word_preds.json` | predicted label for every word, for error analysis (#7) |
| `layoutlm_scores.csv`, `layoutlm_scores_official.csv` | field-level P/R/F1 vs tagged-word gold / official answers |
| `layoutlm_seqeval.txt` | word-level seqeval classification report |
| `layoutlm_hparams.json` | hyperparameters (identical to BERT), GPU and training time |
| `layoutlm_training_log.csv`, `layoutlm_training_loss.png`, `layoutlm_f1.png` | training curve and F1 chart |

No model weights are saved.

In [ ]:
!pip install -q transformers "datasets<3" seqeval

In [ ]:
# Make `src/` importable. In a fresh Colab runtime the repo isn't there yet, so clone it.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/chundawat-h/Receipt-Invoice-Information-Extraction.git"
if Path("../src").is_dir():            # running from notebooks/ inside the repo
    ROOT = Path("..")
elif Path("src").is_dir():
    ROOT = Path(".")
else:                                  # Colab (clone once; re-running the cell reuses it)
    ROOT = Path("Receipt-Invoice-Information-Extraction")
    if not ROOT.is_dir():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
sys.path.insert(0, str(ROOT.resolve()))
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

### Notes from data exploration (#4)
- `bboxes` are **already 0–1000**, so no normalisation is needed. Each sub-token gets its word's box; `[CLS]` → `[0,0,0,0]`, `[SEP]` → `[1000,1000,1000,1000]`, padding → `[0,0,0,0]`.
- BERT and LayoutLM have identical vocabularies, so both models see exactly the same sub-tokens. Use the same hyperparameters as notebook 03.
- Same truncation rule as 03: cut-off words get `"O"`. Save `results/layoutlm_preds.json` keyed by `row["key"]`.

In [ ]:
from src.data import LABEL2ID, LABELS, ID2LABEL, load_sroie

# Word-tagged SROIE (darentang/sroie, Parquet copy): train 626 / test 347.
# Columns: id, words, bboxes (already 0-1000), ner_tags, image_path, key.
# Save predictions keyed by row["key"] (e.g. "X51005230621"), not by row index.
ds = load_sroie()
ds

## 1. Hyperparameters (copied from BERT)

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import (
    AutoTokenizer,
    DataCollatorForTokenClassification,
    LayoutLMForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

# Same values as BERT (#5), read from its saved record so the two can't drift apart
SHARED = ["max_length", "learning_rate", "epochs", "batch_size", "weight_decay", "warmup_steps",
          "lr_scheduler", "optimizer", "fp16", "seed"]
bert_record = json.loads((RESULTS / "bert_hparams.json").read_text())
HP = {"model": "microsoft/layoutlm-base-uncased", **{k: bert_record[k] for k in SHARED}}

EXPECTED = {"max_length": 512, "learning_rate": 5e-5, "epochs": 10, "batch_size": 8, "fp16": True, "seed": 42}
assert all(HP[k] == v for k, v in EXPECTED.items()), f"BERT hyperparameters changed: {HP}"

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
if GPU is None:
    raise RuntimeError("No GPU: in Colab use Runtime -> Change runtime type -> T4 GPU. "
                     "(BERT was trained with fp16 on a T4; LayoutLM must match.)")
print("GPU:", GPU, "| transformers", transformers.__version__, "| torch", torch.__version__)
set_seed(HP["seed"])
pd.DataFrame(HP.items(), columns=["hyperparameter", "value"]).set_index("hyperparameter")

## 2. Tokenise, align labels and add boxes

Labels work exactly as for BERT: only the first sub-token of each word carries the word's label, and everything else gets `-100`.

**New for LayoutLM:** every sub-token also gets a `bbox`:
- Each sub-token gets the box of the word it comes from. The boxes are already 0–1000 (checked in #4), so no normalisation is needed.
- `[CLS]` → `[0, 0, 0, 0]` and `[SEP]` → `[1000, 1000, 1000, 1000]`.
- Padding → `[0, 0, 0, 0]`. This is added by the collator in the next section.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(HP["model"])
MAX_LEN = HP["max_length"]
CLS_BOX, SEP_BOX, PAD_BOX = [0, 0, 0, 0], [1000, 1000, 1000, 1000], [0, 0, 0, 0]
assert tokenizer.padding_side == "right"

def align_labels(word_ids, word_tags):
    # First sub-token of each word -> the word's tag; everything else -> -100 (ignored by the loss)
    labels, previous = [], None
    for w in word_ids:
        labels.append(-100 if w is None or w == previous else word_tags[w])
        previous = w
    return labels

def align_boxes(word_ids, input_ids, word_boxes):
    # Every sub-token gets its word's box; [CLS] -> CLS_BOX, [SEP] -> SEP_BOX
    boxes = []
    for w, token_id in zip(word_ids, input_ids):
        if w is not None:
            boxes.append(list(word_boxes[w]))
        else:
            boxes.append(SEP_BOX if token_id == tokenizer.sep_token_id else CLS_BOX)
    return boxes

def encode(batch):
    enc = tokenizer(batch["words"], is_split_into_words=True, truncation=True, max_length=MAX_LEN)
    enc["labels"], enc["bbox"] = [], []
    for i, (tags, word_boxes) in enumerate(zip(batch["ner_tags"], batch["bboxes"])):
        word_ids = enc.word_ids(i)
        enc["labels"].append(align_labels(word_ids, tags))
        enc["bbox"].append(align_boxes(word_ids, enc["input_ids"][i], word_boxes))
    return enc

encoded = ds.map(encode, batched=True, remove_columns=ds["train"].column_names)

# Sanity checks: one box per sub-token, all inside 0-1000, [CLS]/[SEP] boxes in place
for split in encoded:
    for ex in encoded[split]:
        assert len(ex["bbox"]) == len(ex["input_ids"]) == len(ex["labels"])
        assert ex["bbox"][0] == CLS_BOX and ex["bbox"][-1] == SEP_BOX
        assert all(0 <= v <= 1000 for box in ex["bbox"] for v in box)
print("bbox checks passed")
encoded

In [ ]:
# What the model sees for the start of one receipt: same sub-tokens and labels as BERT, plus a box each
row, enc = ds["train"][0], encoded["train"][0]
word_ids = tokenizer(row["words"], is_split_into_words=True, truncation=True, max_length=MAX_LEN).word_ids()
pd.DataFrame({
    "sub-token": tokenizer.convert_ids_to_tokens(enc["input_ids"]),
    "word": [row["words"][w] if w is not None else "" for w in word_ids],
    "bbox": enc["bbox"],
    "label": [ID2LABEL[l] if l != -100 else "-100 (ignored)" for l in enc["labels"]],
}).head(25)

## 3. Train

The standard `DataCollatorForTokenClassification` pads `input_ids`, `attention_mask` and `labels`, but not `bbox`. The small collator below pads the boxes with `[0, 0, 0, 0]` to the same length.

In [ ]:
def pad_boxes(boxes, length):
    return [list(b) for b in boxes] + [PAD_BOX] * (length - len(boxes))

class LayoutLMCollator:
    def __init__(self, tokenizer):
        self.text = DataCollatorForTokenClassification(tokenizer)

    def __call__(self, features):
        features = [dict(f) for f in features]
        boxes = [f.pop("bbox") for f in features]
        batch = self.text(features)
        length = batch["input_ids"].shape[1]
        batch["bbox"] = torch.tensor([pad_boxes(b, length) for b in boxes], dtype=torch.long)
        return batch

collator = LayoutLMCollator(tokenizer)
sample = collator([encoded["train"][i] for i in range(3)])
print({k: tuple(v.shape) for k, v in sample.items()})

In [ ]:
model = LayoutLMForTokenClassification.from_pretrained(
    HP["model"], num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
)

args = TrainingArguments(
    output_dir="outputs/layoutlm",       # git-ignored; nothing is saved there (save_strategy="no")
    learning_rate=HP["learning_rate"],
    num_train_epochs=HP["epochs"],
    per_device_train_batch_size=HP["batch_size"],
    per_device_eval_batch_size=16,
    weight_decay=HP["weight_decay"],
    warmup_steps=HP["warmup_steps"],
    lr_scheduler_type=HP["lr_scheduler"],
    fp16=HP["fp16"],
    seed=HP["seed"],
    eval_strategy="no",                  # no validation split; test is used once, after training
    save_strategy="no",                  # no checkpoints / weights
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded["train"],
    data_collator=collator,
    processing_class=tokenizer,
)

start = time.time()
train_result = trainer.train()
TRAIN_MINUTES = (time.time() - start) / 60
print(f"Training took {TRAIN_MINUTES:.1f} min on {GPU}")
train_result.metrics

In [ ]:
log = pd.DataFrame([h for h in trainer.state.log_history if "loss" in h])
log.to_csv(RESULTS / "layoutlm_training_log.csv", index=False)

bert_log = pd.read_csv(RESULTS / "bert_training_log.csv")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(bert_log["step"], bert_log["loss"], label="BERT (text only)", alpha=0.7)
ax.plot(log["step"], log["loss"], label="LayoutLM (text + layout)")
ax.set_xlabel("training step")
ax.set_ylabel("training loss")
ax.set_yscale("log")
ax.set_title(f"Training loss ({HP['epochs']} epochs, lr {HP['learning_rate']}, batch {HP['batch_size']})")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS / "layoutlm_training_loss.png", dpi=150)
plt.show()

## 4. Predict on the test set (word level)

This works the same way as BERT: each word takes the prediction of its first sub-token, and words cut off by truncation get `O`.

In [ ]:
def to_word_labels(n_words, word_ids, token_pred):
    labels = ["O"] * n_words          # words cut off by truncation stay "O"
    seen = set()
    for position, w in enumerate(word_ids):
        if w is None or w in seen:
            continue
        seen.add(w)
        labels[w] = ID2LABEL[int(token_pred[position])]
    return labels

test = ds["test"]
token_preds = trainer.predict(encoded["test"]).predictions.argmax(-1)
test_word_ids = [
    tokenizer(r["words"], is_split_into_words=True, truncation=True, max_length=MAX_LEN).word_ids() for r in test
]
word_preds = {
    r["key"]: to_word_labels(len(r["words"]), wid, pred)
    for r, wid, pred in zip(test, test_word_ids, token_preds)
}

with open(RESULTS / "layoutlm_word_preds.json", "w", encoding="utf-8") as f:
    json.dump(word_preds, f)
print(f"Word-level predictions for {len(word_preds)} test receipts saved")

## 5. seqeval report (word level)

In [ ]:
from seqeval.metrics import classification_report

y_true = [[ID2LABEL[t] for t in r["ner_tags"]] for r in test]
y_pred = [word_preds[r["key"]] for r in test]
report = classification_report(y_true, y_pred, digits=4)
(RESULTS / "layoutlm_seqeval.txt").write_text(report)
print(report)

## 6. Field-level scores (`src/evaluate.py`, same as every method)

In [ ]:
from src.evaluate import bio_to_fields, compare, evaluate, get_gold, get_official_gold, load_preds, normalize, save_preds

layoutlm_preds = {r["key"]: bio_to_fields(r["words"], word_preds[r["key"]]) for r in test}
save_preds(layoutlm_preds, RESULTS / "layoutlm_preds.json")

gold_test = get_gold(test)
official_gold = get_official_gold("test", keys=test["key"])   # streams the official answers (~1 min)

print("TEST, main gold (word tags)")
test_scores = evaluate(layoutlm_preds, gold_test)
test_scores.to_csv(RESULTS / "layoutlm_scores.csv")
print(test_scores.to_string())

print("\nTEST, official ICDAR answers")
official_scores = evaluate(layoutlm_preds, official_gold)
official_scores.to_csv(RESULTS / "layoutlm_scores_official.csv")
print(official_scores.to_string())

ceiling = evaluate(gold_test, official_gold)["f1"]

In [ ]:
f1 = pd.DataFrame({"main gold": test_scores["f1"], "official": official_scores["f1"]}).drop("macro_avg")
ax = f1.plot.bar(rot=0, figsize=(7, 3.8), color=["#4C72B0", "#DD8452"])
ax.set_ylim(0, 105)
ax.set_ylabel("F1 (%)")
ax.set_title("LayoutLM (text + layout): test F1 per field")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}", (p.get_x() + p.get_width() / 2, p.get_height() + 1), ha="center", fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS / "layoutlm_f1.png", dpi=150)
plt.show()

## 7. First look: how much does layout add?

This is a quick comparison with the saved regex and BERT predictions, scored by the same `compare()`. `layout_gain` = LayoutLM F1 − BERT F1. The full analysis (chart, error analysis, write-up) is #7.

In [ ]:
method_preds = {
    "Regex": load_preds(RESULTS / "regex_preds.json"),
    "BERT": load_preds(RESULTS / "bert_preds.json"),
    "LayoutLM": layoutlm_preds,
}
print("F1 vs tagged-word gold (main)")
print(compare(method_preds, gold_test).to_string())
print("\nF1 vs official ICDAR answers (perfect-tagger ceiling in the last column)")
official_table = compare(method_preds, official_gold)
official_table["ceiling"] = ceiling
print(official_table.round(2).to_string())

## 8. Where LayoutLM fails (examples for the report)

In [ ]:
bert_preds = method_preds["BERT"]
for field in ["company", "address", "total", "date"]:
    wrong = [k for k, g in gold_test.items() if normalize(layoutlm_preds[k][field], field) != normalize(g[field], field)]
    fixed = [k for k, g in gold_test.items()
             if normalize(layoutlm_preds[k][field], field) == normalize(g[field], field)
             and normalize(bert_preds[k][field], field) != normalize(g[field], field)]
    print(f"=== {field}: LayoutLM wrong on {len(wrong)} / {len(gold_test)} | right where BERT was wrong: {len(fixed)} ===")
    for k in wrong[:3]:
        print(f"  [{k}] PRED: {layoutlm_preds[k][field]!r}\n                 GOLD: {gold_test[k][field]!r}")
    print()

## 9. Summary, hyperparameters and download

In [ ]:
summary = pd.DataFrame({
    "LayoutLM F1, main gold": test_scores["f1"],
    "LayoutLM F1, official": official_scores["f1"],
    "perfect-tagger ceiling (official)": ceiling,
}).round(1)
print(summary.to_string())

record = {
    **HP,
    "gpu": GPU,
    "train_minutes": round(TRAIN_MINUTES, 1),
    "train_receipts": len(ds["train"]),
    "test_receipts": len(test),
    "transformers": transformers.__version__,
    "torch": torch.__version__,
}
(RESULTS / "layoutlm_hparams.json").write_text(json.dumps(record, indent=2))
record

In Colab, the next cell downloads all `results/layoutlm_*` files as one zip.
- **Extract the files straight into the repo's `results/` folder**, not into a `layoutlm_results/` subfolder.
- Save this notebook with **File → Save a copy in GitHub**: branch `model/layoutlm`, file path exactly `notebooks/04_layoutlm.ipynb`, and untick "Include a link to Colaboratory".

In [ ]:
import zipfile

layoutlm_files = sorted(RESULTS.glob("layoutlm_*"))
print("\n".join(p.name for p in layoutlm_files))
try:
    from google.colab import files
except ImportError:
    files = None                      # running locally: the files are already in results/
if files is not None:
    with zipfile.ZipFile("layoutlm_results.zip", "w") as z:
        for p in layoutlm_files:
            z.write(p, p.name)
    files.download("layoutlm_results.zip")